# Test 3 — Trigger por ADC + dead-time y resolución par-pulso

Mide a qué distancia mínima entre pulsos consecutivos el sistema empieza a
perder eventos, **separando las tres escalas que se confunden en una sola
curva**:

| | Piso | Quién lo pone |
|---|---|---|
| HW | ~24 ns | re-arm de `adc_trg_dis` + `trigger_shield` (3 ciclos de ADC) |
| Medición por `Δwp_trig` | 8 ns | 1 sample del buffer (decim=1) |
| Método (polling desde Python) | ~5–25 µs | una vuelta del lazo sobre /dev/mem |

Estrategia:

1. Un **Rigol DG4162** genera el tren de pulsos en IN1; cada flanco ascendente
   es un evento a detectar.
2. Scope con OR_MASK = ADC ch0 posedge (`bit 1`) + `adc_we_keep=1` (continuo) +
   `trigger_shield` con `dur=0` (re-arm por HW, sin pulsar `0x94` desde el SW).
3. **Barrido de período** (1 ms → 1 µs): eficiencia y dt, con `dt_hw` medido por
   `Δwp_trig` y la sonda de `wp_trig` diciendo si el FPGA sigue disparando.
   Mide el piso del *método*.
4. **Diagnóstico** de los puntos que dan `n=0`: generador / camino de trigger /
   polling.
5. **Barrido de ancho** de pulso (posedge vs negedge del mismo pulso): mide la
   resolución par-pulso del *hardware*, sin Python en el lazo.

Todas las capturas se guardan en `datos/<stamp>__<label>.npz` para reanalizar
offline (`datos/plot_datos.py`).

In [ ]:
# === 1) BITSTREAM PRIMERO — antes de rp_Init() y de mapear /dev/mem ===
#
# El orden no es cosmético: mientras la PL se reconfigura no hay esclavo que
# conteste en el bus AXI, así que un acceso posterior a 0x4010_0000 desde un
# mapeo YA ABIERTO da un *external abort* -> SIGBUS -> se muere el proceso.
# En Jupyter se ve como "el kernel murió", sin traceback, y en dmesg como:
#
#   Unhandled fault: external abort on non-linefetch (0x1818) at 0x........
#   [........] *pgd=........, *pte=40100743      <- física 0x40100xxx = scope
#
# Por eso esta celda va ANTES de la de imports (que hace rp_Init + open).
# Para recargar el bitstream a mitad de sesión NO corras esta celda: usá la
# celda "recarga segura" de más abajo, que cierra los mapeos primero.
from multitrigger_utils import load_bitstream, fpga_state

print('estado previo del FPGA manager:', fpga_state())
info = load_bitstream()          # fpgautil -b /root/red_pitaya_top.bit.bin
print(f'bitstream cargado: {info["path"]}   state={info["state"]}')

In [ ]:
# === 2) recién ahora: rp_Init() + mapeo del scope ===
# (la PL ya quedó programada en la celda de arriba; abrir el mapeo con la PL
#  sin programar haría que la primera lectura muera con SIGBUS, por eso
#  MultiTriggerScope.open() chequea fpga0/state antes de mapear)
import os
import time
import numpy as np
from matplotlib import pyplot as plt
import rp

# Driver del scope (arm / captura / diagnóstico) — multitrigger_utils.py
from multitrigger_utils import (
    MultiTriggerScope, decode_snap,
    OR_MASK_ALL,
    BIT_SW, BIT_ADC_P0, BIT_ADC_N0, BIT_ADC_P1, BIT_ADC_N1,
    BIT_EXT_P, BIT_EXT_N, BIT_ASG_P, BIT_ASG_N,
    N_BUF, FS,
    events_to_intervals, efficiency, pulses_from_buffer,
)

# Métodos de caracterización del multitrigger: sondas de HW (trigger_alive),
# análisis de tiempos con reloj de FPGA (wp_dt_us, events_to_hw_intervals)
# y los barridos. Ver testbench_multitrigger.py.
import testbench_multitrigger as tb

rp.rp_Init()
sc = MultiTriggerScope.open()
print(f'cfg sanity: trg_src ch0 @0x240 = {sc.r32(0x240):#010x}  (esperado tras reset: 0)')
print(f'            set_dec ch0 @0x014 = {sc.r32(0x14):#010x}   (esperado tras reset: 1)')
print(f'            calib_gain0 @0x204 = {sc.r32(0x204):#010x}  (esperado tras reset: 0x8000)')

In [ ]:
# --- Guardado de datos crudos para analisis offline ---
# Mismo esquema que multitrigger_test_adq_dg4162.ipynb: cada captura se guarda
# en datos/ como .npz con los arrays + un snapshot de los registros del scope,
# asi se puede reanalizar sin volver a ocupar el HW.
#
# Layout (esquema plano con prefijo por corrida):
#   datos/<AAAAMMDD_HHMMSS>__<label>.npz
#   ej: datos/20260805_153012__dt_p100us.npz
#
# Cargar despues (offline):
#   npz = np.load('datos/<archivo>.npz', allow_pickle=True)
#   dt_hw = npz['dt_hw_us']
#   regs  = npz['meta'].item()        # dict con snapshot, mask, wp_trig, ...

DATA_DIR   = 'datos'
_RUN_STAMP = time.strftime('%Y%m%d_%H%M%S')   # un stamp por corrida del notebook
os.makedirs(DATA_DIR, exist_ok=True)

def scope_regs(sc):
    """Snapshot de los registros clave del scope (para la meta de cada captura)."""
    return dict(
        adc_state=sc.r32(0x00),  trg_state=sc.r32(0x04),
        thr_ch0=sc.r32(0x08),    thr_ch1=sc.r32(0x0C),
        dly_ch0=sc.r32(0x10),    dly_ch1=sc.r32(0x110),
        dec_ch0=sc.r32(0x14),    dec_ch1=sc.r32(0x114),
        hyst_ch0=sc.r32(0x20),   hyst_ch1=sc.r32(0x24),
        shield=sc.r32(0x210),    shield_rt=sc.r32(0x214),
        snapshot=sc.r32(0x218),  dis_we=sc.r32(0x21C),
        mask_ch0=sc.r32(0x240),  mask_ch1=sc.r32(0x244),
        wp_trig0=sc.r32(0x1C),   wp_trig1=sc.r32(0x11C),
        wp_cur0=sc.r32(0x18),    wp_cur1=sc.r32(0x118),
        # we_cnt: samples escritos desde el ultimo arm/reset/pulso a 0x94
        # (NO desde el ultimo trigger) + debouncer del trigger externo
        we_cnt0=sc.r32(0x02C),   we_cnt1=sc.r32(0x12C),
        deb_len=sc.r32(0x90),
    )

def save_capture(label, meta=None, **arrays):
    """Guarda una captura en datos/<stamp>__<label>.npz para analisis offline.

    label : nombre de la etapa (ej. 'dt_p100us', 'dt_sweep_summary').
    arrays: nombre=ndarray, ej. save_capture('x', d1=d1, d2=d2).
    meta  : dict de escalares/registros.
    Devuelve el path escrito.
    """
    os.makedirs(DATA_DIR, exist_ok=True)
    path = os.path.join(DATA_DIR, f'{_RUN_STAMP}__{label}.npz')
    payload = {k: np.asarray(v) for k, v in arrays.items()}
    payload['label']  = label
    payload['fs']     = FS
    payload['t_wall'] = time.time()
    if meta:
        payload['meta'] = np.array(meta, dtype=object)   # dict -> array 0-d (item())
    np.savez_compressed(path, **payload)
    shapes = ', '.join(f'{k}{np.asarray(v).shape}' for k, v in arrays.items())
    print(f'  [save] {path}  <- {shapes}')
    return path

def save_tb(label, meta=None, **arrays):
    """Callback que le pasamos a las rutinas de testbench_multitrigger: agrega
    el snapshot de registros del scope a la meta que arma cada barrido."""
    return save_capture(label, meta={**scope_regs(sc), **(meta or {})}, **arrays)

print(f'guardado listo -> {DATA_DIR}/  (stamp {_RUN_STAMP})')

In [ ]:
# === Recarga del bitstream a mitad de sesión (último recurso) ===
#
# Reconfigurar en caliente NO es seguro del todo en esta placa, y conviene
# saber por qué antes de correr esta celda:
#
#   red_pitaya_top.sv:227   sys_bus_if sys[8] (.clk(adc_clk), .rstn(adc_rstn))
#   red_pitaya_top.sv:289   adc_rstn <= frstn[0] & ~rst_after_locked
#
# o sea que TODO el bus de registros del scope vive en el dominio adc_clk y en
# reset por adc_rstn. Mientras la PL se reconfigura no hay esclavo que conteste
# en 0x4010_0000: el puente GP0 tira *external abort* -> SIGBUS -> muere el
# kernel. Y si el puerto llega a abortar una vez, queda TRABADO: todo acceso
# posterior aborta aunque la PL ya esté bien.
#
#   >>> De ese estado sólo se sale REINICIANDO la Pitaya. <<<
#
# reload_bitstream() hace lo que se puede: cierra el mmap y el rp, baja los
# puentes AXI PS-PL mientras programa (lo que `!fpgautil -b` a secas NO hace),
# espera el lock del PLL y recién ahí reabre. Devuelve una instancia NUEVA, hay
# que reasignar `sc`.
#
# Lo robusto sigue siendo: reiniciar la placa y cargar el bitstream UNA vez en
# la primera celda del notebook, antes de que nada mapee /dev/mem.

sc = MultiTriggerScope.reload_bitstream(scope=sc)
print(sc.verify_bitstream(verbose=True)['ok'])

## Generador externo: Rigol DG4162

La señal de pulsos viene de un **Rigol DG4162** controlado por SCPI vía
USB-TMC (driver kernel `/dev/usbtmcN`) o TCP. El driver está en
[`rigol_dg4162.py`](./rigol_dg4162.py).

Dos modos:

- **Pulsos periódicos** (`set_pulse_periodic(period_s, width_s, ...)`):
  un pulso cada `period_s`. Cada flanco ascendente es un trigger.
  Cambiar el período = cambiar la distancia entre pulsos.
- **Pares de pulsos en burst** (`set_pulse_pair_burst(gap_s, burst_period_s, ...)`):
  dos pulsos separados por `gap_s`, repetidos cada `burst_period_s`.
  Mide directamente la **pulse-pair resolution**.

Cableado: Rigol OUT1 → Pitaya IN1. Jumper LV (1:1).

In [ ]:
# Importar driver del Rigol y abrir conexión.
# Si tu Rigol está en otro device (p.ej. /dev/usbtmc1) o por TCP, cambiá esto.
from rigol_dg4162 import RigolDG4162

#try:
#    rg = RigolDG4162.usbtmc('/dev/usbtmc0')
#except FileNotFoundError:
    # Fallback TCP — poné la IP del Rigol
rg = RigolDG4162.usb()
print('Conectado:', rg.id)

# Configuración base: pulsos angostos (200 ns) de 0 a 1 V.
# Período inicial 100 µs (= 10 kHz). Se cambia con rg.set_pulse_period().
RIGOL_CH        = 1
PULSE_WIDTH_S   = 200e-9
PULSE_AMP_VPP   = 1.0
PULSE_OFFSET_V  = 0.5      # nivel base = 0 V, peak = 1 V

rg.reset()
rg.set_pulse_periodic(ch=RIGOL_CH, period_s=100e-6,
                       width_s=PULSE_WIDTH_S,
                       amp_vpp=PULSE_AMP_VPP, offset_v=PULSE_OFFSET_V)
rg.output(RIGOL_CH, True)
print('Rigol armado: pulsos a 10 kHz, ancho 200 ns, 0–1 V')
print('  último error SCPI:', rg.check_error())

In [ ]:
# === Que se puede testear: limites del estimulo y piso del metodo ===
#
# Antes de barrer nada conviene saber contra que piso estamos midiendo. Son
# tres numeros distintos y hay que no mezclarlos:
#
#   1. LIMITE DEL GENERADOR: ancho/transicion minimos y freq maxima en modo
#      pulso. Se los preguntamos al instrumento (no al datasheet).
#   2. PISO DEL HW: el re-arm del trigger multitrigger es de ~3 ciclos de ADC
#      (~24 ns): adc_trg_dis queda alto 1 ciclo y el trigger_shield con dur=0
#      lo limpia (multitrigger_trig_src.sv + trigger_shield.sv). El
#      adc_wp_trig queda ciego mientras corre el post-trigger delay
#      (rp_bram_sm.v:91), por eso auto_rearm lo deja en 1 sample.
#   3. PISO DEL METODO: cuanto tarda una vuelta del lazo de polling de Python.
#      Los eventos separados por menos que esto se pierden aunque el FPGA los
#      vea. Es el numero que domina todo el barrido de periodos.

lim = tb.rigol_limits(rg, ch=RIGOL_CH)
print('Limites del Rigol (declarados por el instrumento):')
print(f'  ancho      : {lim["width_min"]*1e9:.1f} ns .. {lim["width_max"]*1e3:.3g} ms')
print(f'  transicion : {lim["tran_min"]*1e9:.1f} ns min')
print(f'  frecuencia : {lim["freq_min"]:.3g} .. {lim["freq_max"]:.3g} Hz '
      f'(=> periodo minimo {1e9/lim["freq_max"]:.1f} ns)')

pf = tb.poll_floor_us(sc, n=20000)
print('\nPiso del metodo (lazo de polling sobre /dev/mem):')
print(f'  lectura pura   : {pf["read_us"]:.2f} us')
print(f'  intervalo p50  : {pf["p50"]:.2f} us   p90 = {pf["p90"]:.2f}   p99 = {pf["p99"]:.2f}')
print(f'  => separaciones menores a ~{3*pf["p50"]:.0f} us no se pueden medir con este metodo')
print(f'     (el HW re-arma en ~0.024 us: 3 ordenes de magnitud mas rapido)')

PISO_SW_US = pf['p50']

## Diagnóstico

Si una captura da `capturados: 0` (o cualquier resultado raro), corré esta
celda para ver dónde se corta la cadena. Hace tres pasos:

1. `sc.debug_trigger()` en IDLE → estado base de todos los regs.
2. `sc.acq_capture_sw(thr)` + plot del buffer IN1 → confirma que la señal
   del Rigol llega a la entrada y el `bram_sm` captura un buffer
   consistente (independiente del trigger ADC).
3. `sc.arm_for_adc_trigger(...)` + sleep 100 ms + `sc.debug_trigger()` →
   muestra si el FPGA disparó (snapshot ≠ 0), si la máscara latcheó, si
   we_keep quedó en `0x3` y si `wp_trig` avanzó.

Qué buscar:
- **`snapshot == 0`** tras el arm + sleep → ningún flanco cruzó el threshold.
  Causa: threshold mal, mask mal, señal por debajo del umbral.
- **`dis_act != 0` y quieto en 1** → adc_trg_dis se trabó (problema del
  trig_dis_clr / shield).
- **`we_keep = 0x0`** después del arm → el fix del orden no entró
  (re-cargá `multitrigger_utils.py`).
- **`wp_cur` avanza pero `wp_trig` quieto** → FSM escribe el buffer pero
  el trigger no firma.

In [ ]:
print('=== 1. Estado en IDLE ===')
sc.debug_trigger()

print('\n=== 2. Captura forzada por SW trigger (sanity de buffer + señal) ===')
d1, d2, snap = sc.acq_capture_sw(thr=0.5)
print(f'  snap = {snap:#010x} -> {decode_snap(snap)}   (esperado: sw_any)')
print(f'  IN1 pp = {d1.max()-d1.min():.3f} V   media = {d1.mean():+.3f} V')
print(f'  IN1 max = {d1.max():.3f} V   min = {d1.min():.3f} V   '
      f'(esperado pp ~1 V, mean ~0.5 V con offset Rigol)')

plt.figure(figsize=(10, 3))
plt.plot(d1[:2000], label='IN1 (Rigol)')
plt.axhline(0.5, color='r', ls='--', lw=0.8, label='threshold 0.5 V')
plt.xlabel('sample'); plt.ylabel('V')
plt.title(f'SW trigger sanity — pp={d1.max()-d1.min():.2f} V')
plt.legend(); plt.grid(True); plt.show()

print('\n=== 3. Arm ADC trigger ch0 posedge, esperar 100 ms ===')
sc.arm_for_adc_trigger(mask_ch0=BIT_ADC_P0, mask_ch1=BIT_ADC_P0, thr=0.5)
print('  estado JUSTO después del arm:')
sc.debug_dump()
time.sleep(0.1)
print('\n  estado DESPUÉS de 100 ms:')
sc.debug_trigger()
sc.disarm()

## Captura de N eventos consecutivos

Las funciones viven en [`multitrigger_utils.py`](./multitrigger_utils.py) y se
usan vía la instancia `sc` (`MultiTriggerScope`):

- `sc.arm_for_adc_trigger(mask_ch0, mask_ch1, thr, hyst, delay, we_keep_both, auto_rearm)`:
  configura decim/threshold/delay/hyst + OR_MASK + arm + we_keep.

  **`auto_rearm=True`** (default): activa el `trigger_shield` para clear
  automático del `adc_trg_dis` (shield_dur=0, mismo ciclo) Y escribe
  `set_dly` raw (saltea el offset de ~65 µs que mete
  `rp_AcqSetTriggerDelay`). Con esto el dead-time del stack baja a la
  región de pocos µs.

  **`auto_rearm=False`**: comportamiento legacy. El SW tiene que pulsar
  `0x94` entre triggers y el `bram_sm` tiene el offset de 65 µs.

  **`delay`**: post-trigger delay, en samples. Es dead-time REAL — mientras
  corre, `adc_dly_do` bloquea la actualización de `adc_wp_trig`
  (`rp_bram_sm.v:91`) — así que para medir dead-time va en el mínimo
  (`delay=0` → 1 sample). Subirlo sólo para ventanas post-trigger
  (`capture_window_np`).

  **OJO**: la API `rp_AcqStart` borra el `we_keep` como side-effect (escribe
  byte0=0x01 en 0x00, lo que en la cfg satisface `|sys_dats` y resetea el
  bit 3). Por eso el método setea we_keep DESPUÉS del arm.

- `sc.capture_n_events(n, timeout_ms, wp_addr, clear_both, read_snap, deadline_every)`:
  polea cambios de `wp_trig @0x1C`. Con `auto_rearm=True` el shield limpia
  `adc_trg_dis` por HW, así que el loop solo lee `wp_trig`. **El costo de una
  vuelta de ese loop es el dead-time del método**: `read_snap=False` (default)
  ahorra la lectura de 0x218 por evento y el deadline se chequea cada
  `deadline_every` vueltas, no en cada una.

- `sc.disarm()`: apaga we_keep, desactiva el shield, limpia
  `adc_trg_dis` y resetea el FSM.

- `events_to_intervals(events)`: intervalos SW (µs) entre triggers.
- `efficiency(observed_n, target_freq, duration)`: ratio capturados / esperados.

Y en [`testbench_multitrigger.py`](./testbench_multitrigger.py) (importado como
`tb`), lo específico de caracterización:

- `tb.events_to_hw_intervals(events)`: los mismos intervalos pero medidos con
  `Δwp_trig`, o sea con el reloj del FPGA (**8 ns** de resolución en vez de los
  ±4 µs del reloj de Python).
- `tb.trigger_alive(sc)`: sonda de HW sobre `wp_trig` @0x1C, que se re-latchea
  en cada disparo aceptado. Dice si el FPGA sigue disparando aunque el SW no
  vea todos los eventos, y de paso da `dt_hw_med_us` (Δwp entre cambios).
  (`tb.we_time_since_arm_us` lee `adc_we_cnt` @0x2C, que **no** se resetea en
  cada disparo: el `trig_dis_clr` que llega al `bram_sm` es el del registro
  0x94, no el del `trigger_shield` — ver `rp_scope_multitrigger_com.sv:597`.)
- `tb.poll_floor_us(sc)`: cuánto tarda una vuelta del lazo de polling.
- `tb.pulse_metrics(d)`: amplitud/ancho/período de lo que realmente llega al ADC.

## Ejemplo: una captura de N eventos

Rigol genera pulsos cada 100 µs en OUT1 (= 10 kHz, ancho 200 ns).
Threshold del scope a 0.5 V (mitad de la altura del pulso).
Esperamos un trigger por cada pulso → dt esperado = 100 µs.

**Si `capturados = 0`**, correr la celda **Diagnóstico** de arriba para ver
dónde se rompe (señal, mask, threshold, we_keep).

In [ ]:
PERIOD_S = 100e-6   # 10 kHz → 100 µs entre pulsos
rg.set_pulse_period(ch=RIGOL_CH, period_s=PERIOD_S)
# auto_rearm=True (default) activa el trigger_shield para clear automático
# del adc_trg_dis + sobreescribe set_dly raw (saltea el offset de 65 µs
# que mete rp_AcqSetTriggerDelay). Sin esto el dead-time floor es ~65 µs.
sc.arm_for_adc_trigger(mask_ch0=BIT_ADC_P0, mask_ch1=BIT_ADC_P0,
                        thr=0.5, auto_rearm=True)

# El FPGA, ¿está disparando? (sonda de HW: wp_trig @0x1C se re-latchea en cada
# disparo aceptado; los Δwp entre cambios dan el dt con 8 ns de resolución)
alive = tb.trigger_alive(sc, dwell_s=0.05)

events, dur_s = sc.capture_n_events(n=50, timeout_ms=1000, read_snap=True)
sc.disarm()

intervals = events_to_intervals(events)           # reloj de Python  (±µs)
dt_hw     = tb.events_to_hw_intervals(events)     # reloj del FPGA   (8 ns)
expected_dt = PERIOD_S * 1e6
print(f'capturados: {len(events)} eventos en {dur_s*1e3:.1f} ms')
print(f'HW: alive={alive["alive"]}  cambios={alive["n_cambios"]}  '
      f'dt_hw={alive["dt_hw_med_us"]:.3f} µs  '
      f'dis_act={alive["dis_act"]:#x}  we_keep={alive["we_keep"]:#x}')
if len(intervals):
    print(f'dt SW (µs): media={intervals.mean():.2f}  std={intervals.std():.2f}  '
          f'min={intervals.min():.2f}  max={intervals.max():.2f}')
    print(f'dt HW (µs): media={dt_hw.mean():.3f}  std={dt_hw.std():.3f}  '
          f'(Δwp_trig, resolución 8 ns)')
    print(f'esperado    : {expected_dt:.2f} µs')
    print(f'eficiencia  : {efficiency(len(events), 1.0/PERIOD_S, dur_s)*100:.1f}%')
snaps = set(e['snap'] for e in events)
print(f'snapshots únicos: {snaps} -> {[decode_snap(s) for s in snaps]}')

save_tb(f'dt_single_p{PERIOD_S*1e6:.0f}us',
        meta={'period_s': PERIOD_S, 'thr': 0.5, 'duration_s': dur_s,
              'observed_n': len(events), 'hw_alive': alive['alive']},
        t_ns=np.array([e['t_ns'] for e in events], dtype=np.int64),
        wp=np.array([e['wp'] for e in events], dtype=np.int32),
        snap=np.array([e['snap'] for e in events], dtype=np.int64),
        dt_sw_us=intervals, dt_hw_us=dt_hw)

if len(intervals):
    plt.figure(figsize=(10, 3))
    plt.plot(intervals, 'o-', label='dt SW (reloj Python)')
    plt.plot(dt_hw, 's-', alpha=0.8, label='dt HW (Δwp_trig)')
    plt.axhline(expected_dt, color='r', ls='--', label=f'esperado {expected_dt:.1f} µs')
    plt.xlabel('evento #'); plt.ylabel('dt entre triggers (µs)')
    plt.legend(); plt.grid(True); plt.title(f'Captura N=50, Rigol @{PERIOD_S*1e6:.0f} µs')
    plt.show()

## Barrido de distancia entre pulsos

`tb.sweep_periods(sc, rg, periods_s, ...)` (en
[`testbench_multitrigger.py`](./testbench_multitrigger.py)). Por cada distancia
esperada (`period_s`):

1. Reconfigurar el Rigol y **verificar qué quedó puesto** (`rigol_state`: si el
   instrumento recortó el período o el ancho, o dejó un error SCPI, se ve acá).
2. Captura cruda por SW trigger + `pulse_metrics` → **qué llega realmente a la
   entrada del ADC** (amplitud, ancho, período medidos sobre el buffer).
3. Armar y preguntarle al FPGA si dispara (`trigger_alive`, sonda sobre `wp_trig`).
4. Capturar N eventos poleando `wp_trig` y medir eficiencia.

Cada punto reporta **tres** cosas que antes se confundían en una:

| campo | qué es |
|---|---|
| `efficiency`, `mean_dt_us` | lo que ve el **software** (reloj de Python, ±4 µs) |
| `hw_dt_med_us` | dt real entre disparos por `Δwp_trig` — resolución **8 ns** |
| `n_saltados_med` | disparos que el FPGA hizo y el polling **no llegó a ver** |
| `hw_alive` | si el FPGA seguía disparando durante el punto |

La distancia más chica con eficiencia ~100 % es el **piso del método**
(polling desde Python), no el del FPGA: para eso está el barrido de ancho de
más abajo.

In [ ]:
# Barrido de distancias entre pulsos: de 1 ms (fácil) a 1 µs (extremo).
# Cada elemento es el período entre pulsos en segundos.
#
# sweep_periods / plot_deadtime_curve viven en testbench_multitrigger.py; acá
# sólo se los llama con el scope, el generador y el callback de guardado.
PERIODS = [1e-3, 500e-6, 200e-6, 100e-6, 50e-6, 20e-6,
           10e-6, 5e-6, 2e-6, 1e-6]

# width_s NO es opcional: el DG4000 conserva el DUTY al cambiar la frecuencia,
# así que si no se lo re-escribe en cada punto el ancho se achica junto con el
# período (medido: quedaba en 1 % del período -> 20 ns a p=2 µs, que ya no
# cruza el umbral). Sin esto el barrido mide el ancho de banda de la entrada
# creyendo que mide dead-time.
results = tb.sweep_periods(sc, rg, PERIODS, ch=RIGOL_CH,
                           n_events=300, timeout_ms=2000, thr=0.5,
                           width_s=PULSE_WIDTH_S, raw=True, save=save_tb)
tb.plot_deadtime_curve(results)

## Cómo interpretar la curva

Tres paneles, tres preguntas distintas:

- **Eficiencia vs distancia**: plateau en 100 % a distancias largas, caída a
  distancias chicas. La caída marca dónde el **polling** deja de seguir el
  ritmo — salvo que el punto aparezca con una cruz roja (`hw_alive=False`), en
  cuyo caso el FPGA directamente no está disparando y hay que ir al
  diagnóstico de abajo.
- **dt medido vs esperado**: `dt_hw` (Δwp_trig) tiene que caer sobre la
  diagonal `y=x` mientras el FPGA siga cada pulso. Si `dt_sw` se despega hacia
  arriba y `dt_hw` sigue pegado a la diagonal, **el dead-time es del software**.
  Si `dt_hw` se va a 2× o 3× la diagonal, ahí sí se están perdiendo disparos en
  el FPGA.
- **Disparos perdidos entre observaciones**: cuántos pulsos pasaron entre dos
  lecturas consecutivas de `wp_trig` (`round(dt_hw/período) − 1`). En 0
  mientras el método aguanta; sube apenas el período baja del piso de polling.

**Los tres pisos, para no confundirlos** (ver la celda de límites al principio):

| | Piso | Quién lo pone |
|---|---|---|
| HW | **~24 ns** | re-arm de `adc_trg_dis` + `trigger_shield` (3 ciclos de ADC) |
| Medición por `Δwp_trig` | **8 ns** | 1 sample del buffer (decim=1) |
| Método (polling) | **~5–25 µs** | una vuelta del lazo de Python sobre /dev/mem |

Es decir: con tren periódico + polling no se puede testear nada por debajo de
unas decenas de µs, por más que se achique el período. Para llegar al piso real
del hardware hay que sacar a Python del lazo → el barrido de ancho de la
sección siguiente.

## Diagnóstico de los puntos que dan `n = 0`

Un punto con `n=0` **no** es "el sistema es lento": por lento que sea el lazo,
`wp_trig` cambia entre polls y se contarían igual cientos de eventos (con el dt
sobreestimado). `n=0` durante 2 s significa que `wp_trig` **no cambió nunca**.

`tb.diagnose_point(sc, rg, p)` recorre la cadena y dictamina:

| señal en IN1 | FPGA (`trigger_alive`) | SW | conclusión |
|---|---|---|---|
| sin pulsos / `v_max` < thr | — | n=0 | **generador**: recortó ancho/período o quedó en error SCPI |
| pulsos OK | muerto | n=0 | **camino de trigger**: mirar `dis_act`, `we_keep`, `mask`, `adc_state` |
| pulsos OK | vivo | n=0 | **polling**: el FPGA dispara, el lazo no lo ve |
| pulsos OK | vivo | n < esperado | **dead-time SW**, cuantificado por `dt_hw` |

In [ ]:
# Diagnóstico de los períodos que se cortaron en el barrido.
# Cada llamada guarda su .npz (señal cruda incluida) para revisar offline.
diags = [tb.diagnose_point(sc, rg, p, ch=RIGOL_CH, thr=0.5,
                           width_s=PULSE_WIDTH_S, save=save_tb)
         for p in (10e-6, 5e-6, 2e-6, 1e-6)]

print('\n=== resumen ===')
for d in diags:
    print(f'p={d["period_s"]*1e6:>7.2f} µs  n={d["observed_n"]:>4d}  '
          f'v_max={d["pulso"]["v_max"]:.3f} V  '
          f'ancho={d["pulso"]["ancho_med_ns"]:>7.1f} ns  '
          f'hw_alive={str(d["hw"]["alive"]):>5s}  -> {d["veredicto"]}')

# Con width_s el ancho se re-escribe en cada punto, así que el estímulo es el
# mismo en todo el barrido. Si aun así el ancho MEDIDO EN EL ADC no da ~200 ns
# o v_max queda por debajo del umbral, el generador no obedeció: mirá
# `rigol_width` en la meta del .npz antes de culpar al FPGA.

## Resolución par-pulso real — barrido de **ancho** de pulso

El barrido de período mide el piso del *método* (~decenas de µs). Para llegar
al piso del *hardware* hay que sacar a Python del lazo, y para eso el truco es
usar **los dos flancos del mismo pulso**:

> un pulso de ancho `W` genera un flanco de subida y uno de bajada separados
> **exactamente** por `W`.

Armando con `OR_MASK = BIT_ADC_P0 | BIT_ADC_N0`, el `trig_snapshot` (@0x218)
guarda **cuál fue el último evento aceptado**:

- `adc_n0` → se aceptaron los dos flancos separados por `W` ⇒ **resuelto**;
- `adc_p0` → el flanco de bajada cayó dentro del dead-time ⇒ **no resuelto**.

El generador corre a 1 kHz (tasa cómoda): el resultado **no depende** de la
velocidad del polling, que es justamente el punto. Cross-check independiente:
si se aceptan los dos flancos, los `Δwp_trig` dejan de ser múltiplos exactos
del período y aparece un residuo de ±`W` (`resid_med_ns`).

El `W` donde el snapshot cambia de `adc_n0` a `adc_p0` **es** la resolución
par-pulso del stack. Esperado ~25–30 ns: el re-arm de la lógica son ~24 ns
(3 ciclos de ADC) y el resto lo pone el ancho de banda analógica de la entrada
(~50 MHz ⇒ un pulso de menos de ~25 ns no llega a amplitud plena). Por eso el
umbral baja a 0.3 V y hay que mirar `v_max` de cada punto: si el codo coincide
con el pulso apagándose, el límite es analógico y no dead-time.

In [ ]:
# Barrido de ancho: de 2 µs (trivialmente resuelto) al mínimo del generador.
# El límite inferior sale de lo que declara el propio Rigol (celda de límites).
W_MIN = max(lim['width_min'], 10e-9)
WIDTHS = [2e-6, 1e-6, 500e-9, 200e-9, 100e-9, 50e-9, 30e-9, 20e-9, W_MIN]
WIDTHS = sorted({w for w in WIDTHS if w >= W_MIN}, reverse=True)

wres = tb.sweep_pulse_width(sc, rg, WIDTHS, ch=RIGOL_CH,
                            period_s=1e-3, thr=0.3, hyst=0.01, save=save_tb)
tb.plot_width_curve(wres)

# Restaurar el pulso original por si se sigue usando el notebook
rg.set_pulse_periodic(ch=RIGOL_CH, period_s=100e-6, width_s=PULSE_WIDTH_S,
                      amp_vpp=PULSE_AMP_VPP, offset_v=PULSE_OFFSET_V)

In [ ]:
try:
    sc.disarm()
except Exception as e:
    print('warn disarm:', e)
try:
    rg.output(RIGOL_CH, False)
    rg.close()
except Exception as e:
    print('warn rigol:', e)
sc.close()
rp.rp_Release()
print('cerrado')